# 1 · Digitising the transcriptions (OCR)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/01_ocr/01_ocr_digitisation.ipynb)

**Pipeline stage 1 of 6 — Kölsch phoneme recognition.** Convert the printed *Alles Kölsch*
(Bhatt & Lindlar, 1998) transcription pages into clean, machine-readable
Unicode text, preserving the dialectal orthography.

We evaluate three OCR engines in increasing order of robustness and keep the
best output:

1. **Tesseract** — open-source baseline (offline, free).
2. **EasyOCR** — deep-learning OCR, better on mixed fonts.
3. **Gemini 2.5 Pro** — vision-language model that reads pages *in context*;
   selected for production because it preserves elision apostrophes
   (`d'r`, `m'r`, `ha'mer`), dialectal spellings and special characters.

> Input: a folder of scanned page images. Output: a mirrored folder of `.txt`
> files with the faithful transcription.

## Setup

In [ ]:
# Colab / local install
!pip -q install pytesseract easyocr opencv-python google-generativeai pillow
# Tesseract binary + German language pack (Colab/Ubuntu)
!apt-get -qq install -y tesseract-ocr tesseract-ocr-deu >/dev/null
import os, glob, pathlib
from PIL import Image
print("ready")

In [ ]:
# Point this at your scanned pages (one image per page).
PAGES_DIR  = "pages"          # input: .png/.jpg scans
OUT_DIR    = "ocr_txt"        # output: .txt per page
os.makedirs(OUT_DIR, exist_ok=True)
# For a quick demo with no data, drop a couple of images into ./pages first.
sample = sorted(glob.glob(f"{PAGES_DIR}/*.png") + glob.glob(f"{PAGES_DIR}/*.jpg"))
print(len(sample), "page images found")

## 1 · Tesseract (baseline, with OpenCV preprocessing)

Offline and free. Raw scans of this book truncate line-ends (page curvature) and
Tesseract drops the `ï` diacritic, so we first **upscale → denoise →
adaptive-threshold** the page with OpenCV and run the LSTM engine in single-block
mode (`--psm 6`). This recovers faint print and the full line width; `process_page`
saves the result to `ocr_txt/tesseract_<page>.txt`.

In [ ]:
import os, pathlib, cv2, pytesseract
from PIL import Image

OUT_DIR = "ocr_txt"
os.makedirs(OUT_DIR, exist_ok=True)

# preprocessing knobs
UPSCALE = 2
DENOISE_STRENGTH = 10
ADAPTIVE_BLOCK_SIZE = 35
ADAPTIVE_C = 15
TESS_CONFIG = "--oem 1 --psm 6"   # LSTM engine; assume one uniform block of text

def preprocess_for_tesseract(img_path, out_path=None,
                             upscale=UPSCALE,
                             denoise_strength=DENOISE_STRENGTH,
                             block_size=ADAPTIVE_BLOCK_SIZE,
                             c=ADAPTIVE_C):
    """Upscale -> denoise -> adaptive threshold. Recovers faint print and stops
    page curvature from truncating the ends of lines. Returns the cleaned image
    (or its path if out_path is given)."""
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {img_path}")
    upscaled = cv2.resize(img, None, fx=upscale, fy=upscale,
                          interpolation=cv2.INTER_CUBIC)
    denoised = cv2.fastNlMeansDenoising(upscaled, h=denoise_strength)
    binarized = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=block_size,
        C=c,
    )
    if out_path:
        cv2.imwrite(out_path, binarized)
    return out_path if out_path else binarized

def ocr_tesseract(img_path, lang="deu", config=TESS_CONFIG):
    return pytesseract.image_to_string(Image.open(img_path), lang=lang, config=config)

def process_page(img_path, out_dir=OUT_DIR):
    stem = pathlib.Path(img_path).stem
    cleaned_path = os.path.join(out_dir, f"{stem}_tess_cleaned.png")
    preprocess_for_tesseract(img_path, cleaned_path)
    text = ocr_tesseract(cleaned_path)
    out_path = os.path.join(out_dir, f"tesseract_{stem}.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    print("saved ->", out_path)
    return text

# demo on one page (put your scan at pages/page_1.png)
sample = ["pages/page_1.png"]
if sample and os.path.exists(sample[0]):
    text = process_page(sample[0])
    print(text[:500])
else:
    print("Drop a page image at", sample[0], "to run the demo.")

## 2 · EasyOCR (with geometry-based line reconstruction)

EasyOCR is a deep OCR model, but its native reading order is unreliable on dense
pages. We keep only confident word boxes (conf ≥ 0.4), then rebuild lines from
box geometry — group boxes that overlap vertically, order each line
left-to-right — and save to `ocr_txt/easyocr_<page>.txt`.

In [ ]:
import easyocr
reader = easyocr.Reader(["de"], gpu=True)   # set gpu=False on CPU-only machines

def ocr_easyocr(img_path, y_overlap_ratio=0.5):
    """EasyOCR returns unordered word boxes. We group boxes into lines by
    vertical overlap, then order each line left-to-right, so the text reads in
    natural order (EasyOCR's own reading order is unreliable on dense pages)."""
    results = reader.readtext(img_path, paragraph=False, detail=1)
    results = [r for r in results if r[2] >= 0.4]   # drop low-confidence junk
    boxes = []
    for bbox, text, conf in results:
        xs = [p[0] for p in bbox]
        ys = [p[1] for p in bbox]
        boxes.append({
            "text": text,
            "x0": min(xs), "x1": max(xs),
            "y0": min(ys), "y1": max(ys),
            "h": max(ys) - min(ys),
        })
    boxes.sort(key=lambda b: b["y0"])
    lines = []
    for b in boxes:
        placed = False
        for line in lines:
            line_y0 = min(x["y0"] for x in line)
            line_y1 = max(x["y1"] for x in line)
            overlap = min(b["y1"], line_y1) - max(b["y0"], line_y0)
            min_h = min(b["h"], line_y1 - line_y0) or b["h"]
            if overlap > min_h * y_overlap_ratio:
                line.append(b)
                placed = True
                break
        if not placed:
            lines.append([b])
    lines.sort(key=lambda line: min(x["y0"] for x in line))
    out_lines = []
    for line in lines:
        line.sort(key=lambda b: b["x0"])
        out_lines.append(" ".join(b["text"] for b in line))
    return "\n".join(out_lines)

# demo (reuses `sample` from the Tesseract cell)
if sample and os.path.exists(sample[0]):
    text = ocr_easyocr(sample[0])
    print(text[:500])
    out_path = os.path.join(OUT_DIR, f"easyocr_{pathlib.Path(sample[0]).stem}.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    print("saved ->", out_path)

## 3 · Gemini 2.5 Pro  (selected)

A vision-language model reads each page *with contextual understanding* rather
than glyph-by-glyph, so it keeps apostrophes, dialectal spellings and special
characters. The prompt instructs the model to transcribe **faithfully** and to
**not** correct or normalise the dialect — the standard failure mode of
LLM-based OCR on non-standard text.

Get a key at <https://aistudio.google.com/app/apikey> and set it below.

In [ ]:
import google.generativeai as genai
from google.colab import userdata  # on Colab; else use os.environ

# GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "PASTE_YOUR_KEY")
genai.configure(api_key=GEMINI_API_KEY)

OCR_PROMPT = (
    "You are an OCR system for German dialect (Kölsch) text. "
    "Read all text from this image and output clean Unicode text. "
    "Do NOT translate or correct the dialect spelling. "
    "Preserve apostrophes, special characters, paragraph and line breaks. "
    "Do NOT explain anything. Output the text as plain text only."
)

def ocr_gemini(img_path, model_name="gemini-2.5-pro"):
    model = genai.GenerativeModel(model_name)
    img = Image.open(img_path)
    resp = model.generate_content([OCR_PROMPT, img])
    return resp.text

if sample:
    print(ocr_gemini(sample[0])[:500])

## 4 · Batch driver — process the whole page tree with Gemini

In [ ]:
import time

def batch_ocr(pages_dir=PAGES_DIR, out_dir=OUT_DIR, engine=ocr_gemini, overwrite=False):
    imgs = sorted(glob.glob(f"{pages_dir}/**/*.png", recursive=True) +
                  glob.glob(f"{pages_dir}/**/*.jpg", recursive=True))
    for i, img in enumerate(imgs, 1):
        rel = os.path.relpath(img, pages_dir)
        out = os.path.join(out_dir, os.path.splitext(rel)[0] + ".txt")
        os.makedirs(os.path.dirname(out), exist_ok=True)
        if os.path.exists(out) and not overwrite:
            continue
        try:
            text = engine(img)
            with open(out, "w", encoding="utf-8") as f:
                f.write(text)
            print(f"[{i}/{len(imgs)}] {rel}  ({len(text)} chars)")
        except Exception as e:
            print(f"[{i}/{len(imgs)}] {rel}  ERROR: {e}")
        time.sleep(0.5)   # be gentle with the API
    print("done ->", out_dir)

# batch_ocr()   # uncomment to run over ./pages

## 5 · Final human pass

OCR is never perfect on dialect text. After the batch run, a human proofreads
each `.txt` against the source page — correcting residual errors and confirming
that elisions and special characters survived. The corrected `.txt` tree is the
input to **Notebook 2 (corpus statistics)** and **Notebook 3 (segmentation)**.

### Why Gemini was selected
| Engine | Strength | Weakness on Kölsch |
|---|---|---|
| Tesseract | offline, free | drops `'` elisions, mishandles diacritics |
| EasyOCR | good on layout | weak on dialect spellings / special chars |
| **Gemini 2.5 Pro** | contextual, keeps orthography | needs an API key (small cost) |
